<h1>WWTW CNN — Model Comparison</h1>

Loads the pickled results from each training notebook (`results_<component>_resnet18.pkl`, `results_<component>_resnet50.pkl`, etc.) and builds the comparison table + bar charts. Doesn't need a GPU, torch, or any of the trained models in memory — just the small pickled metrics/history dicts.

Set `COMPONENT` below to whichever component you want to compare (`aerobic_zone` or `clarifier`) — results are saved per-component, so training both won't overwrite each other, but you do need to pick one here to compare.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")

COMPONENT = "clarifier"   # or "aerobic_zone" — must match what you trained

RESULT_FILES = {
    "ResNet-18": f"results_{COMPONENT}_resnet18.pkl",
    "ResNet-50": f"results_{COMPONENT}_resnet50.pkl",
    "DenseNet-121": f"results_{COMPONENT}_densenet121.pkl",
    "InceptionNet v3": f"results_{COMPONENT}_inception_v3.pkl",
    "EfficientNet-B0": f"results_{COMPONENT}_efficientnet_b0.pkl",
    "ConvNeXt-Tiny": f"results_{COMPONENT}_convnext_tiny.pkl",
}

# Missing files are skipped gracefully below, so this still works fine if
# you have not trained every architecture yet.

## Load the pickled results

In [ ]:
results = {}
for model_name, filename in RESULT_FILES.items():
    path = RESULTS_DIR / filename
    if not path.exists():
        print(f"Missing: {path} — run the corresponding training notebook first.")
        continue
    with open(path, "rb") as f:
        results[model_name] = pickle.load(f)

print(f"Loaded results for: {list(results.keys())}")

## Comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "model": name,
        "test_accuracy": r["test_acc"],
        "mean_auc": r["mean_auc"],
        "hidden_layers": r["hidden_layers"],
        "neurons": r["neurons"],
        "img_size": r["img_size"],
    }
    for name, r in results.items()
]).set_index("model")

display(comparison)

## Comparison bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(comparison.index, comparison["test_accuracy"], color="#4C72B0")
axes[0].set_title(f"Test accuracy — {COMPONENT}")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(comparison.index, comparison["mean_auc"], color="#55A868")
axes[1].set_title(f"Mean AUC — {COMPONENT}")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Training curves for every model

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(12, 4 * len(results)))
if len(results) == 1:
    axes = axes.reshape(1, -1)

for row, (name, r) in enumerate(results.items()):
    history = r["history"]
    best_epoch = int(np.argmin(history["val_loss"])) + 1

    axes[row, 0].plot(history["train_loss"], label="train")
    axes[row, 0].plot(history["val_loss"], label="val")
    axes[row, 0].axvline(best_epoch - 1, color="gray", linestyle="--", linewidth=1)
    axes[row, 0].set_title(f"{name} — Loss")
    axes[row, 0].legend()

    axes[row, 1].plot(history["train_acc"], label="train")
    axes[row, 1].plot(history["val_acc"], label="val")
    axes[row, 1].axvline(best_epoch - 1, color="gray", linestyle="--", linewidth=1)
    axes[row, 1].set_title(f"{name} — Accuracy")
    axes[row, 1].legend()

plt.tight_layout()
plt.show()